# 02 · Threading in Depth: Synchronisation Primitives, Thread Safety & Patterns

**Module 13 – Concurrency & asyncio** · Estimated time: 60 min

> Lesson 01 gave the overview. Here: every tool in the `threading` module and how to write correct threaded code.

### 🎯 What you will learn
- `Thread` in detail: subclassing, names, daemon threads, `join(timeout)`, exceptions in threads
- `Lock` vs `RLock`, **deadlocks** and how to avoid them
- `Event`, `Condition`, `Semaphore`/`BoundedSemaphore`, `Barrier`, `Timer`
- Thread-safe queues: `Queue`, `LifoQueue`, `PriorityQueue`, 🆕 `Queue.shutdown()` (3.13)
- `threading.local()` and `contextvars`
- What's atomic in CPython (and why you still need locks)
- Graceful shutdown patterns

In [1]:
import threading, time, random, queue
print("main thread:", threading.current_thread().name, "| active threads:", threading.active_count())

main thread: MainThread | active threads: 8


## 1. Threads in detail

In [2]:
class Downloader(threading.Thread):            # subclass style (target= style shown in lesson 01)
    def __init__(self, url):
        super().__init__(name=f"dl-{url}")
        self.url, self.result = url, None
    def run(self):                              # runs in the new thread
        time.sleep(random.uniform(0.05, 0.2))
        self.result = f"{self.url}: {random.randint(100, 999)} bytes"

random.seed(1)
workers = [Downloader(u) for u in ("a.com", "b.com", "c.com")]
for w in workers: w.start()
for w in workers: w.join(timeout=2)            # never wait forever in production code
print([w.result for w in workers], [w.is_alive() for w in workers])

['a.com: 361 bytes', 'b.com: 607 bytes', 'c.com: 220 bytes'] [False, False, False]


### Daemon threads
A **daemon** thread is killed abruptly when the main program exits, which is fine for background helpers like heartbeats, but never for work that must finish (writing files, DB commits).

In [3]:
def heartbeat(stop: threading.Event):
    while not stop.wait(0.1):          # wait() doubles as an interruptible sleep
        pass

stop = threading.Event()
hb = threading.Thread(target=heartbeat, args=(stop,), daemon=True, name="heartbeat")
hb.start()
print(hb.daemon, hb.name, hb.is_alive())
stop.set(); hb.join()
print("heartbeat stopped:", not hb.is_alive())

True heartbeat True


heartbeat stopped: True


### Exceptions in threads don't reach the main thread

In [4]:
def failing():
    raise ValueError("boom in a thread")

errors = []
threading.excepthook = lambda args: errors.append(f"{args.thread.name}: {args.exc_value!r}")   # global hook
t = threading.Thread(target=failing, name="worker-1")
t.start(); t.join()
threading.excepthook = threading.__excepthook__          # restore the default
print("captured:", errors)

captured: ["worker-1: ValueError('boom in a thread')"]


✅ Easier: use `concurrent.futures` (lesson 04), where `future.result()` **re-raises** the worker's exception in the caller.

## 2. Locks

### `Lock`: mutual exclusion (recap) and the `with` form

In [5]:
class Counter:
    def __init__(self):
        self.value = 0
        self._lock = threading.Lock()
    def increment(self):
        with self._lock:                 # acquire ... release, even on exceptions
            self.value += 1

c = Counter()
threads = [threading.Thread(target=lambda: [c.increment() for _ in range(10_000)]) for _ in range(8)]
for t in threads: t.start()
for t in threads: t.join()
print(c.value)

80000


`lock.acquire(timeout=1)` returns `False` instead of waiting forever; `lock.locked()` checks the state.

### `RLock`: a lock the same thread can acquire again

In [6]:
class Inventory:
    def __init__(self):
        self.items = {}
        self._lock = threading.RLock()          # re-entrant
    def add(self, name, qty):
        with self._lock:
            self.items[name] = self.items.get(name, 0) + qty
    def add_many(self, entries):
        with self._lock:                         # holds the lock...
            for name, qty in entries:
                self.add(name, qty)              # ...and add() acquires it AGAIN: fine with RLock, deadlock with Lock

inv = Inventory()
inv.add_many([("pen", 3), ("book", 1), ("pen", 2)])
print(inv.items)

{'pen': 5, 'book': 1}


### Deadlocks

Two threads each hold one lock and wait for the other's: both wait forever.

```
Thread 1: with lock_a: ... with lock_b: ...
Thread 2: with lock_b: ... with lock_a: ...      ← opposite order → possible deadlock
```
✅ Rules: **acquire locks in a consistent global order**; hold locks briefly; never call unknown code (callbacks) while holding a lock; use timeouts to detect problems.

In [7]:
lock_a, lock_b = threading.Lock(), threading.Lock()

def transfer(first, second, name):
    with first:
        time.sleep(0.05)
        if second.acquire(timeout=0.3):          # timeout → we detect the deadlock instead of hanging
            second.release()
            print(f"{name}: done")
        else:
            print(f"{name}: ⚠️ could not get the second lock (would deadlock)")

t1 = threading.Thread(target=transfer, args=(lock_a, lock_b, "T1"))
t2 = threading.Thread(target=transfer, args=(lock_b, lock_a, "T2"))   # opposite order!
t1.start(); t2.start(); t1.join(); t2.join()

T2: ⚠️ could not get the second lock (would deadlock)T1: ⚠️ could not get the second lock (would deadlock)



## 3. `Event`: one-time signals between threads

In [8]:
ready = threading.Event()

def server():
    time.sleep(0.2)                  # pretend to start up
    print("server: ready")
    ready.set()

def client(i):
    ready.wait()                     # block until the event is set
    print(f"client {i}: connected")

threads = [threading.Thread(target=client, args=(i,)) for i in range(3)] + [threading.Thread(target=server)]
for t in threads: t.start()
for t in threads: t.join()

server: ready
client 0: connected
client 1: connected
client 2: connected


## 4. `Condition`: wait until some state is true

In [9]:
class BoundedBuffer:
    def __init__(self, capacity):
        self.items, self.capacity = [], capacity
        self.cond = threading.Condition()
    def put(self, item):
        with self.cond:
            self.cond.wait_for(lambda: len(self.items) < self.capacity)   # releases the lock while waiting
            self.items.append(item)
            self.cond.notify_all()
    def get(self):
        with self.cond:
            self.cond.wait_for(lambda: self.items)
            item = self.items.pop(0)
            self.cond.notify_all()
            return item

buf, got = BoundedBuffer(2), []
producer = threading.Thread(target=lambda: [buf.put(i) for i in range(6)])
consumer = threading.Thread(target=lambda: [got.append(buf.get()) for _ in range(6)])
producer.start(); consumer.start(); producer.join(); consumer.join()
print(got)

[0, 1, 2, 3, 4, 5]


(That's how `queue.Queue` works internally; normally just use `Queue`.)

## 5. `Semaphore`: limit concurrent access

In [10]:
pool = threading.BoundedSemaphore(2)             # at most 2 threads inside at once
active, peak, lock = 0, 0, threading.Lock()

def use_connection(i):
    global active, peak
    with pool:
        with lock:
            active += 1; peak = max(peak, active)
        time.sleep(0.1)
        with lock:
            active -= 1

threads = [threading.Thread(target=use_connection, args=(i,)) for i in range(6)]
for t in threads: t.start()
for t in threads: t.join()
print("max simultaneous users:", peak)

max simultaneous users: 2


`BoundedSemaphore` raises if released more times than acquired, which catches bugs that plain `Semaphore` hides.

## 6. `Barrier`: wait until N threads reach the same point

In [11]:
barrier = threading.Barrier(3, action=lambda: print("--- all players ready, start! ---"))
def player(name):
    time.sleep(random.uniform(0, 0.2))
    print(f"{name} ready")
    barrier.wait()
    print(f"{name} playing")

threads = [threading.Thread(target=player, args=(n,)) for n in ("A", "B", "C")]
for t in threads: t.start()
for t in threads: t.join()

C ready
B ready
A ready
--- all players ready, start! ---
A playing
C playing
B playing


## 7. `Timer`: run something later (cancellable)

In [12]:
t = threading.Timer(0.2, lambda: print("reminder fired"))
t.start()
cancelled = threading.Timer(0.2, lambda: print("never printed"))
cancelled.start(); cancelled.cancel()
t.join()

reminder fired


## 8. Queues: the safest way to share work

| Class | Order |
|---|---|
| `queue.Queue` | FIFO |
| `queue.LifoQueue` | LIFO (stack) |
| `queue.PriorityQueue` | lowest priority first (`(priority, item)` tuples) |
| `queue.SimpleQueue` | unbounded FIFO, fewer features, re-entrant |

🆕 Python 3.13 added **`Queue.shutdown()`**: consumers blocked on `get()` receive `queue.ShutDown` instead of needing sentinel values.

In [13]:
jobs = queue.PriorityQueue()
results = []

def worker():
    while True:
        try:
            priority, job = jobs.get()
        except queue.ShutDown:               # 🆕 3.13: clean exit
            return
        results.append((priority, job, threading.current_thread().name))
        jobs.task_done()

for prio, job in [(3, "send newsletter"), (1, "charge card"), (2, "resize image")]:
    jobs.put((prio, job))
workers = [threading.Thread(target=worker, name=f"w{i}") for i in range(2)]
for w in workers: w.start()
jobs.join()                                  # wait for all task_done()
jobs.shutdown()                              # wake up idle workers so they can exit
for w in workers: w.join()
print(sorted(results))

[(1, 'charge card', 'w0'), (2, 'resize image', 'w0'), (3, 'send newsletter', 'w0')]


## 9. Thread-local data and `contextvars`

`threading.local()` gives each thread its **own** copy of an attribute (e.g. a DB connection or request info):

In [14]:
local = threading.local()

def handle(request_id):
    local.request_id = request_id            # each thread sees only its own value
    time.sleep(0.05)
    return f"thread {threading.current_thread().name} handled {local.request_id}"

from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(3, thread_name_prefix="req") as ex:
    print(*ex.map(handle, ["r1", "r2", "r3"]), sep="\n")

thread req_0 handled r1
thread req_1 handled r2
thread req_2 handled r3


`contextvars.ContextVar` does the same but also works correctly with **asyncio tasks** (lesson 06), so prefer it in new code.

## 10. What's thread-safe without a lock?

In CPython (with the GIL), single operations on built-ins like `list.append`, `dict[key] = value`, `deque.append`/`popleft` are atomic. **Compound** operations are not:

| Safe-ish (single op) | ❌ Needs a lock (read-modify-write) |
|---|---|
| `lst.append(x)` | `counter += 1` |
| `d[k] = v`, `d.get(k)` | `if k not in d: d[k] = []` (check-then-act) |
| `q.put(x)` / `q.get()` (Queue) | `x = lst[-1]; lst.pop()` |

> ⚠️ Don't rely on these details: they're CPython implementation behaviour, and on the **free-threaded** build (lesson 08) races appear more often. When in doubt, use a `Lock` or a `Queue`.

## 11. Graceful shutdown pattern

In [15]:
class Service:
    def __init__(self, n_workers=2):
        self.stop = threading.Event()
        self.tasks = queue.Queue()
        self.workers = [threading.Thread(target=self._run, name=f"svc-{i}") for i in range(n_workers)]
    def start(self):
        for w in self.workers: w.start()
    def _run(self):
        while not self.stop.is_set():
            try:
                task = self.tasks.get(timeout=0.1)      # wake up regularly to check `stop`
            except queue.Empty:
                continue
            print(f"{threading.current_thread().name} processed {task}")
            self.tasks.task_done()
    def shutdown(self):
        self.tasks.join()                               # finish queued work
        self.stop.set()
        for w in self.workers: w.join()

svc = Service()
svc.start()
for i in range(4):
    svc.tasks.put(f"job{i}")
svc.shutdown()
print("clean shutdown; threads alive:", [w.is_alive() for w in svc.workers])

svc-0 processed job0svc-1 processed job1
svc-1 processed job2
svc-1 processed job3

clean shutdown; threads alive: [False, False]


## 📝 Summary
- `Thread(target=..)` or subclass `run()`; `join(timeout)`; daemon threads die with the program.
- `Lock`/`RLock` for mutual exclusion (consistent order avoids deadlocks); `Event` for signals; `Condition` for "wait until"; `Semaphore` to limit access; `Barrier` to synchronise phases; `Timer` for delays.
- Prefer **queues** for passing work; 🆕 `Queue.shutdown()` (3.13).
- `threading.local`/`contextvars` for per-thread data; compound operations always need locks.

## 🏋️ Practice
1. Implement a thread-safe `RateLimiter(max_calls, period)` using a `Lock` and timestamps.
2. Build a pipeline of three thread stages (download → parse → save) connected by queues, shut down with `Queue.shutdown()`.
3. Reproduce a deadlock with two locks, then fix it by enforcing a lock order.